# Web-Based RAG System
## Retrieval-Augmented Generation from Web Pages




## Setup: Import Libraries and Load API Key

In [21]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Check if API key is set
if os.environ.get('GROQ_API_KEY'):
    print("GROQ_API_KEY is set")
else:
    print("GROQ_API_KEY is not set")
    

GROQ_API_KEY is set


In [22]:
import requests
# import re
from bs4 import BeautifulSoup

# using groq
from langchain_groq import ChatGroq

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document


# import faiss
# i am working with chroma db
from langchain_community.vectorstores import Chroma 
import numpy as np



## Step 1: Configure URLs to Index , making document

In [23]:
#defining the urls

urls = [
    "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://en.wikipedia.org/wiki/Car",
    "https://en.wikipedia.org/wiki/Life"
]

print(f"no. of URLs:  {len(urls)}")
# for i, url in enumerate(urls, 1):
#     print(f"  {i}. {url}")

no. of URLs:  4


#### clean 
Reduces noise - Removes irrelevant content that would confuse embeddings

Saves tokens - Less text means lower API costs for LLM calls

In [24]:
def fetch_and_clean(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    response = requests.get(url, headers=headers)  # #fetch the content of the url
    
    soup = BeautifulSoup(response.text, "html.parser")   #raw html --> parse

    
    # Remove unwanted tags
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.extract()
    
    text = soup.get_text() #extract the cleaned text , removing the html tags
    return text

In [25]:
# def fetch_and_clean(url):
#     headers = {
#         'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
#     }
#     response = requests.get(url) #fetch the content of the url
#     soup = BeautifulSoup(response.text, "html.parser") #raw html --> parse

#     # Remove unwanted tags
#     for tag in soup(["script", "style", "nav", "footer", "header"]):
#         tag.extract()

#     text = soup.get_text() #extract the cleaned text , removing the html tags
#     return text

#### convert to document

In [26]:
documents = [] #empty list

for url in urls:
    text = fetch_and_clean(url) # Gets cleaned text content from the webpage
    
    doc = Document(
        page_content=text,
        metadata={"source": url}   
    )
    
    documents.append(doc)

### step 2 ; CHUNKING

In [27]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

### step 3 : Embeddings

In [28]:
# paid
# from langchain_core import OpenAIEmbeddings


# free
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2" 
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6091.47it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
embeddings.embed_query("What is the AI?")
# simply, embeddding vannale vector ma change garne process ho. text lai vector ma convert garne process ho.
# embedding is a way to represent text in a numerical format that captures its meaning and context.


[-0.04706783592700958,
 -0.007012251298874617,
 -0.020528987050056458,
 0.01450338028371334,
 0.039129648357629776,
 0.004470204468816519,
 0.055067650973796844,
 0.022081956267356873,
 0.014159667305648327,
 0.06518200784921646,
 -0.03954856097698212,
 -0.016401022672653198,
 0.026982583105564117,
 -0.06526146829128265,
 -0.0969155803322792,
 0.03169538825750351,
 0.02320438250899315,
 -0.0823872834444046,
 -0.08305767178535461,
 -0.07384362071752548,
 0.009576152078807354,
 0.02244941145181656,
 -0.013898362405598164,
 -0.048225950449705124,
 -0.04017505422234535,
 0.07896218448877335,
 3.4823471651179716e-05,
 -0.060777757316827774,
 -0.00338098150677979,
 -0.015261762775480747,
 0.03146770969033241,
 -0.009374866262078285,
 0.10357300937175751,
 0.014604561030864716,
 -0.09146644175052643,
 0.03796845301985741,
 -0.029939424246549606,
 -0.03373653069138527,
 0.07457568496465683,
 -0.006638480816036463,
 0.006644427310675383,
 -0.06657417118549347,
 0.02015487663447857,
 -0.07439758

#### step 4 : Create and Store Embeddings in Vector Store.


In [30]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    
    )

# if used faiss, then the code would be like this

# from langchain.vectorstores import FAISS

# vectorstore = FAISS.from_documents(chunks, embeddings)

## LLM

In [31]:

#  llm=ChatOpenAI(model="gpt-5-nano", temperature=0)
# for groq

# llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

llm = ChatGroq(
    temperature=0,
    model_name="llama-3.1-8b-instant"   
)

#### semantic search

In [32]:
context = vectorstore.similarity_search("RAG",k=3)  

In [33]:
##### What's happening internally:
# query = "RAG"
# query_embedding = embeddings.embed_query(query)  # Convert "RAG" to vector
#similarities = vectorstore.index.search(query_embedding, k=3)  # Find nearest neighbors
##### Returns: [Document1, Document2, Document3] (most relevant first)

In [34]:
response=llm.invoke(f"what is AI? You can answer using the following context: {context}")
print(response.content)

# Creates a prompt that includes the retrieved context
# Sends it to the LLM (like Groq's Mixtral)
# The LLM generates an answer based on both its training and the provided context

Based on the provided context, it appears that the source is Wikipedia, and the topic is related to machine learning and categories. However, to provide a clear and concise answer, I'll use general knowledge to explain what AI is.

**Artificial Intelligence (AI)**

Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:

1. **Learning**: AI systems can learn from data, identify patterns, and improve their performance over time.
2. **Reasoning**: AI systems can make decisions based on logic, rules, and experience.
3. **Problem-solving**: AI systems can solve complex problems, often in a more efficient and effective way than humans.
4. **Perception**: AI systems can interpret and understand data from sensors, such as images, speech, and text.

AI involves various disciplines, including:

1. **Machine learning**: a subset of AI that enables systems to learn from data and improve their performance

### sends: Python objects with metadata and brackets , llm --> confused

#### better approach

## Queryy

In [ ]:
def ask_question(query, k=3):
    docs = vectorstore.similarity_search(query, k=k)
    
    print("\n Retrieved Evidence:\n")
    
    context = ""
    for i, doc in enumerate(docs):
        print(f"--- Chunk {i+1} ---") # Print chunk number
        print(f"Source: {doc.metadata.get('source')}") # Print source URL
        print(doc.page_content[:300]) # Print the first 300 characters of the chunk
        print("\n")
        context += doc.page_content + "\n" # Combine all retrieved chunks into a single context string
    
    prompt = f"""Answer the question based ONLY on the context below:{context} Question: {query} """
    
    response = llm.invoke(prompt)  # Use invoke() 
    
    print(" Answer:\n")
    print(response.content)  # Access .content attribute

In [40]:
ask_question("what is artificial intelligence   ?")


 Retrieved Evidence:

--- Chunk 1 ---
Source: https://en.wikipedia.org/wiki/Artificial_intelligence
rarely described as "artificial intelligence" (a tendency known as the AI effect).[381]


--- Chunk 2 ---
Source: https://en.wikipedia.org/wiki/Artificial_intelligence
rarely described as "artificial intelligence" (a tendency known as the AI effect).[381]


--- Chunk 3 ---
Source: https://en.wikipedia.org/wiki/Artificial_intelligence
Artificial intelligence - Wikipedia


























Jump to content












































From Wikipedia, the free encyclopedia



Intelligence of machines
"AI" redirects here. For other uses, see AI (disambiguation) and Artificial intelligence (disambiguation).


 Answer:

According to the provided context, artificial intelligence refers to the "intelligence of machines".
